# Refract: Model Evaluation with Deterministic Ground Truth

This notebook shows how to use Refract to evaluate AI models for temporal leakage,
retrieval quality, and provenance hallucination.

**No API keys needed.** Refract is deterministic — same input, same output, every time.

In [ ]:
!pip install refract-py pandas matplotlib -q

In [ ]:
# Install the Refract CLI (required by refract-py)
!npm install -g @refract-org/cli 2>/dev/null || echo "CLI already installed or using npx fallback"

## 1. Analyze a page and load events

**No API key needed.** First run fetches from Wikipedia (requires internet).
Subsequent runs use the local cache.

If the CLI isn't available, the next cell loads a pre-computed sample.

In [ ]:
# Try live analysis, fall back to pre-computed sample
import pandas as pd
import json
from urllib.request import urlopen

try:
    from refract import Refract
    r = Refract()
    df = r.analyze("COVID-19", depth="forensic", as_frame=True)
    print(f"Loaded {len(df)} events from live Wikipedia analysis")
except Exception as e:
    print(f"CLI not available ({e}), loading sample data...")
    url = "https://raw.githubusercontent.com/refract-org/refract-demo-data/main/historical-wikipedia/sample-covid-events.jsonl"
    events = []
    with urlopen(url) as f:
        for line in f:
            line = line.decode().strip()
            if line and not line.startswith("#"):
                events.append(json.loads(line))
    # Build a minimal DataFrame matching Refract's flat format
    rows = []
    for e in events:
        fact = e.get("deterministicFacts", [{}])[0]
        rows.append({
            "timestamp": e.get("timestamp"),
            "event_type": e.get("eventType"),
            "from_revision_id": e.get("fromRevisionId"),
            "to_revision_id": e.get("toRevisionId"),
            "section": e.get("section"),
            "fact": fact.get("fact", ""),
            "fact_detail": fact.get("detail", ""),
            "layer": e.get("layer"),
            "event_id": e.get("eventId", ""),
        })
    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} events from sample dataset")

print(f"Event types: {df['event_type'].nunique()}")
df.head()

## 2. Temporal leakage detection

Find claims that first appeared after a model's training cutoff.

In [ ]:
from datetime import datetime

CUTOFF = datetime(2024, 6, 1)  # GPT-4o cutoff

first_seen = df[df["event_type"] == "sentence_first_seen"].copy()
first_seen["timestamp_dt"] = pd.to_datetime(first_seen["timestamp"])

leaked = first_seen[first_seen["timestamp_dt"] > CUTOFF]
known = first_seen[first_seen["timestamp_dt"] <= CUTOFF]

print(f"Claims before cutoff: {len(known)}")
print(f"Claims after cutoff:  {len(leaked)}")
print(f"Leakage candidates:   {len(leaked)} claims a model should not know")
print()
print("Sample claims after cutoff:")
for _, row in leaked.head(5).iterrows():
    text = row.get("fact_detail", "")[:120]
    print(f"  [{row['timestamp'][:10]}] {text}")

## 3. Retrieval quality scoring

Score claims by stability for use as RAG retrieval weights.

In [ ]:
# Score each claim by contestation signals
stability = df.groupby("fact_detail").agg(
    reverts=("event_type", lambda x: (x == "revert_detected").sum()),
    citation_churn=("event_type", lambda x: (x.str.startswith("citation_")).sum()),
    talk_activity=("event_type", lambda x: (x.str.startswith("talk_")).sum()),
    first_seen=("timestamp", "min"),
).reset_index()

stability["contestation_score"] = (
    stability["reverts"] * 0.4 +
    stability["citation_churn"] * 0.1 +
    stability["talk_activity"] * 0.05
)
stability["stability"] = 1.0 - stability["contestation_score"].clip(0, 1)

print(f"Mean stability: {stability['stability'].mean():.3f}")
print(f"Contested claims (stability < 0.7): {(stability['stability'] < 0.7).sum()}")

# Most contested claims
stability.sort_values("contestation_score", ascending=False).head(10)[
    ["fact_detail", "reverts", "citation_churn", "talk_activity", "stability"]
]

## 4. Provenance hallucination check

Did a source ever exist on this page?

In [ ]:
def check_source(source_pattern: str) -> dict:
    citations = df[df["event_type"].str.startswith("citation_")]
    matches = citations[
        citations["fact_detail"].str.contains(source_pattern, na=False)
    ]
    
    verified = len(matches[matches["event_type"] == "citation_added"])
    removed = len(matches[matches["event_type"] == "citation_removed"])
    replaced = len(matches[matches["event_type"] == "citation_replaced"])
    
    if verified + removed + replaced == 0:
        return {"status": "hallucinated", "message": f"No citation matching '{source_pattern}' ever found"}
    if removed > 0 or replaced > 0:
        return {"status": "outdated", "added": verified, "removed": removed, "replaced": replaced}
    return {"status": "verified", "count": verified}

# Test a known source
print(check_source("who.int"))

# Test a likely hallucinated source
print(check_source("nonexistent-source-xyz.gov"))

## 5. Export for benchmark submission

Submit results to the community benchmark.

In [ ]:
import json

benchmark = {
    "model": "your-model-name",
    "claimed_cutoff": "2024-06-01",
    "evaluated_at": pd.Timestamp.now().isoformat(),
    "refract_version": "0.5.3",
    "results": {
        "total_claims_tested": len(first_seen),
        "claims_after_cutoff": len(leaked),
        "leakage_rate": round(len(leaked) / max(len(first_seen), 1), 4),
        "mean_stability": round(stability["stability"].mean(), 3),
    }
}

with open("benchmark_results.json", "w") as f:
    json.dump(benchmark, f, indent=2)

print(json.dumps(benchmark, indent=2))
print("\nSaved to benchmark_results.json")
print("Submit to: https://github.com/refract-org/refract/blob/main/BENCHMARK.md")